# Top Scorer in Each Group

## Tables

**`players`** — contains the player ID and the group each player belongs to.

| Column | Type |
| --- | --- |
| `player_id` | int |
| `group_id` | int |

**`matches`** — each row records a match between two players and their respective scores.

| Column | Type |
| --- | --- |
| `match_id` | int |
| `first_player` | int |
| `second_player` | int |
| `first_score` | int |
| `second_score` | int |

## Question

Find the **top scorer in each group**. A player's total score is the sum of all scores they earned across every match (whether they appeared as `first_player` or `second_player`). If two players in the same group are tied on total score, the one with the **lower `player_id`** wins.

Return `group_id`, `player_id`, and `score`.

In [0]:
%sql
-- ============================================================
-- Cell 1: Create and populate the `players` and `matches` tables
-- ============================================================
create table b_sql.b_practice.players
(player_id int,
group_id int);

insert into b_sql.b_practice.players values (15,1);
insert into b_sql.b_practice.players values (25,1);
insert into b_sql.b_practice.players values (30,1);
insert into b_sql.b_practice.players values (45,1);
insert into b_sql.b_practice.players values (10,2);
insert into b_sql.b_practice.players values (35,2);
insert into b_sql.b_practice.players values (50,2);
insert into b_sql.b_practice.players values (20,3);
insert into b_sql.b_practice.players values (40,3);

create table b_sql.b_practice.matches
(
match_id int,
first_player int,
second_player int,
first_score int,
second_score int);

insert into b_sql.b_practice.matches values (1,15,45,3,0);
insert into b_sql.b_practice.matches values (2,30,25,1,2);
insert into b_sql.b_practice.matches values (3,30,15,2,0);
insert into b_sql.b_practice.matches values (4,40,20,5,2);
insert into b_sql.b_practice.matches values (5,35,50,1,1);


In [0]:
%sql
-- ============================================================
-- Cell 2: Preview the players table
-- ============================================================
select  * from b_sql.b_practice.players;


In [0]:
%sql
-- ============================================================
-- Cell 3: Preview the matches table
-- ============================================================
select * from b_sql.b_practice.matches

In [0]:
%sql
-- ============================================================
-- Cell 4: SQL solution — find the top scorer in each group
-- Unions first/second player scores, sums per player, joins to
-- players for group_id, and uses QUALIFY + RANK() to pick the
-- highest scorer per group (ties broken by lower player_id).
-- ============================================================
with score as (select first_player as player ,first_score as score from b_sql.b_practice.matches
union all
select second_player as player ,second_score as score from b_sql.b_practice.matches),
player_score as(
select player, sum(score) as total_score from score group by player order by total_score desc),
player_group as (select player, total_score,group_id from b_sql.b_practice.players p join player_score ps on p.player_id = ps.player order by total_score desc)
select *  from player_group qualify rank() over(partition by group_id order by total_score desc,player)=1


In [0]:
# ============================================================
# Cell 5: Load matches and players tables into PySpark DataFrames
# ============================================================
df_match= spark.read.table("b_sql.b_practice.matches")
df_player= spark.read.table("b_sql.b_practice.players")
df_match.display()
df_player.display()

In [0]:
# ============================================================
# Cell 6: Import PySpark SQL functions, types, and Window
# (Window is needed for ranking in Cell 7)
# ============================================================
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
# ============================================================
# Cell 7: PySpark solution — find the top scorer in each group
# Unions first/second player scores, sums per player, joins to
# players for group_id, and uses row_number() over a Window
# partitioned by group_id to pick the highest scorer per group.
# ============================================================
df_first=df_match.select(col("first_player").alias("player"),col("first_score").alias("score"))
df_second=df_match.select(col("second_player").alias("player"),col("second_score").alias("score"))
df_union=df_first.union(df_second).groupBy("player").agg(sum("score").alias("score")).orderBy(col("score").desc())
df_join=df_player.join(df_union,df_player.player_id==df_union.player,"inner").select(col("group_id"),col("player"),col("score")).orderBy(col("score").desc())
df_join.display()
w=Window.partitionBy(col("group_id")).orderBy(col("score").desc(),col("player"))
df_final=df_join.withColumn("rnk",row_number().over(w)).filter(col("rnk")==1)
df_final.display()